# 🔷 OpenAI Agents SDK — Lightweight, Opinionated Orchestration

> *"Four primitives, production-ready, minimal abstraction."*

**By OpenAI** · Evolved from **Swarm** (2025) · The most *opinionated* mainstream framework

---

## Table of Contents
1. [What is the OpenAI Agents SDK?](#1)
2. [The Four Primitives](#2)
3. [How Handoffs Work](#3)
4. [How Guardrails Work](#4)
5. [Sessions, Sandboxes & Tracing](#5)
6. [Conceptual Code](#6)
7. [How the OpenAI Agents SDK is Advanced vs Other Frameworks](#7)
8. [Strengths, Weaknesses & When to Use](#8)


<a id="1"></a>
## 1. What is the OpenAI Agents SDK?

The **OpenAI Agents SDK** is a **lightweight, production-ready** framework for building multi-agent workflows on OpenAI models. It grew out of OpenAI's experimental **Swarm** project and shipped as a supported SDK in early 2025.

Its design choice: **be opinionated.** Rather than offer endless flexibility, it gives you a **small set of well-designed primitives** that cover most agent needs — fewer decisions, faster to ship.

> Because it sits directly on OpenAI's models, it gets **native function calling, structured outputs, and streaming with no abstraction layer** in between.


<a id="2"></a>
## 2. The Four Primitives

The entire SDK is built on just **four** concepts:

```
┌─────────────┐   ┌─────────────┐   ┌─────────────┐   ┌─────────────┐
│   AGENTS    │   │    TOOLS    │   │  HANDOFFS   │   │ GUARDRAILS  │
│ LLM + instr.│   │ functions   │   │ delegate to │   │ input/output│
│  + tools    │   │ agents call │   │ other agents│   │ safety checks│
└─────────────┘   └─────────────┘   └─────────────┘   └─────────────┘
```

| Primitive | What it is |
|---|---|
| **Agents** | An LLM configured with instructions and a set of tools |
| **Tools** | Functions an agent can call to act on the world |
| **Handoffs** | One agent **delegating** the task to another, more specialized agent |
| **Guardrails** | Validation checks on **inputs and outputs** (safety + cost control) |

Master these four and you can express most agent systems.


<a id="3"></a>
## 3. How Handoffs Work

**Handoffs** let an agent **transfer control to a specialist** — the heart of OpenAI's multi-agent model.

```
              ┌────────────────┐
   user  ───▶ │  Triage Agent  │
              └───────┬────────┘
            decides where to route
        ┌─────────────┼─────────────┐
        ▼             ▼             ▼
 ┌────────────┐ ┌────────────┐ ┌────────────┐
 │ Refund     │ │ Order      │ │ FAQ        │
 │ Agent      │ │ Status Ag. │ │ Agent      │
 └────────────┘ └────────────┘ └────────────┘
```

The clever part: **handoffs are represented to the LLM as tools.** A handoff to a "Refund Agent" appears as a callable tool named `transfer_to_refund_agent`. The model "calls" it, and control moves — clean, native, and easy to reason about.


<a id="4"></a>
## 4. How Guardrails Work

**Guardrails** validate inputs and outputs — for safety *and* cost. They run as **input guardrails** (before execution), **output guardrails** (after), and even **tool guardrails** (on every tool call).

Two execution modes give a real engineering tradeoff:

| Mode | Behavior | Best for |
|---|---|---|
| **Parallel** (`run_in_parallel=True`, default) | Guardrail runs **concurrently** with the agent | **Lowest latency** |
| **Blocking** (`run_in_parallel=False`) | Guardrail completes **before** the agent runs; a tripwire stops the agent entirely | **Cost control** — never spend tokens / trigger side effects on bad input |

> A tripped guardrail can **prevent the agent from ever running**, which avoids both token spend *and* risky tool side effects — a thoughtful production detail.


<a id="5"></a>
## 5. Sessions, Sandboxes & Tracing

Beyond the four primitives, the SDK ships production essentials:

| Feature | What it gives you |
|---|---|
| **Sessions** | A persistent memory layer that maintains working context within an agent loop |
| **Sandbox agents** | Run specialists inside **real isolated workspaces** (manifest-defined files, resumable sessions) |
| **Human-in-the-loop** | Built-in mechanisms to involve humans across agent runs |
| **Tracing** | Built-in visualization, debugging, and monitoring — plus hooks into OpenAI's **eval, fine-tuning, and distillation** tools |

The **tracing** integration is a standout: it ties directly into OpenAI's evaluation and fine-tuning suite, so you can debug *and* improve agents in one ecosystem.


<a id="6"></a>
## 6. Conceptual Code

```python
from agents import Agent, Runner, function_tool, GuardrailFunctionOutput

# 1. A tool is just a decorated function
@function_tool
def get_order_status(order_id: str) -> str:
    return orders_db[order_id]["status"]

# 2. Specialist agents
refund_agent = Agent(name="Refund Agent",
                     instructions="Handle refund requests politely and verify eligibility.")
status_agent = Agent(name="Order Status Agent",
                     instructions="Look up and report order status.",
                     tools=[get_order_status])

# 3. A triage agent that HANDS OFF to specialists
triage = Agent(
    name="Triage Agent",
    instructions="Route the customer to the right specialist.",
    handoffs=[refund_agent, status_agent],   # become transfer_to_* tools
)

# 4. Run — the SDK manages routing, tools, and tracing
result = Runner.run_sync(triage, "Where is my order #4821?")
print(result.final_output)
```

> The SDK turns each handoff target into a `transfer_to_<agent>` tool automatically, so routing is just the model "calling a tool."


<a id="7"></a>
## 7. How the OpenAI Agents SDK is Advanced vs Other Frameworks

| Dimension | OpenAI Agents SDK | LangGraph | CrewAI | Claude Agent SDK |
|---|---|---|---|---|
| **Philosophy** | **Opinionated, minimal** (4 primitives) | Maximal control | Role-based teams | Model-native autonomy |
| **Multi-agent** | **Handoffs** (agent-as-tool) | Graph topologies | Crews/Flows | Compose your own |
| **Safety** | **First-class guardrails** (parallel/blocking) | DIY | DIY | Permission gates |
| **Native model features** | **Zero-abstraction** function calling/streaming | Via adapters | Via adapters | Native (Claude) |
| **Tracing/evals** | **Integrated** with OpenAI eval/fine-tune suite | LangSmith | Crew traces | Vendor tracing |
| **Lock-in** | OpenAI models | Open | Open | Anthropic models |

### What makes it "advanced"
1. **Opinionated minimalism.** Four primitives cover most needs, so teams ship faster with **fewer architectural decisions** — the simplicity *is* the feature.
2. **Handoffs-as-tools** is an elegant multi-agent model: routing reuses the LLM's existing tool-calling skill instead of a separate orchestration layer.
3. **Guardrails with parallel/blocking modes** bake in a real latency-vs-cost tradeoff — production safety that would otherwise take weeks to build.
4. **Zero-abstraction + integrated tracing/evals.** Direct access to OpenAI model features, plus tracing wired into eval/fine-tuning, makes the *measure → improve* loop tight.


<a id="8"></a>
## 8. Strengths, Weaknesses & When to Use

### ✅ Strengths
- **Fastest to ship** if you're already on OpenAI — opinionated and minimal
- Clean **agent-to-agent handoffs**
- **Guardrails + tracing** primitives save weeks of custom work
- Native model features with no abstraction overhead

### ⚠️ Weaknesses
- **Tied to OpenAI models** — least portable of the frameworks here
- Opinionated by design → **less flexible** for unusual control flows than LangGraph
- Fewer third-party integrations than the LangChain ecosystem

### 🎯 Use the OpenAI Agents SDK when…
- Your stack is **already on OpenAI** and you want minimal friction
- You need clean **multi-agent handoffs** + built-in **guardrails/tracing**
- You prefer **fewer decisions** and a fast, opinionated path to production

> **Rule of thumb:** *If you're on OpenAI and want clean handoffs with safety and tracing out of the box, this is the shortest path — the opinionation saves you time.*
